Notebook to convert the Aqemia databases in a format acceptable by RxnFlow 
- .sdf or .smi or .smi.gz for the building blocks
- .txt for the reactions

In [13]:
import gzip
import multiprocessing
from pathlib import Path

from awswrangler import s3
from tqdm import tqdm

from _a_refine import get_clean_smiles

NUM_CPUS = 8

### Building blocks : parquet to `.smi.gz`

In [14]:
NAME = "2024_11_simple_bbs_24K"
path = f"s3://aqemia-datahub/forward_synthesis/building_blocks/{NAME}/cleaned/data.parquet"
bb_df = s3.read_parquet(path)
print(f"Loaded {len(bb_df)} building blocks. Columns: {list(bb_df.columns)}")
bb_df.head()

Loaded 23789 building blocks. Columns: ['Mw', 'IUPAC_Name', 'URL', 'Stock_weight_G', 'Class', 'Subclass', 'SMILES', 'has_salt', 'ID', 'num_heavy', 'mol_weight', 'num_H_donors', 'logP', 'num_rotatable_bonds', 'tpsa', 'is_selected', 'num_aromatic', 'num_stereocenter', 'to_remove_nico_rule', 'num_stereoisomer']


,Mw,IUPAC_Name,URL,Stock_weight_G,Class,Subclass,SMILES,has_salt,ID,num_heavy,mol_weight,num_H_donors,logP,num_rotatable_bonds,tpsa,is_selected,num_aromatic,num_stereocenter,to_remove_nico_rule,num_stereoisomer
0,143.57,2-chloro-1-(1H-pyrrol-2-yl)ethan-1-one,https://www.enaminestore.com/catalog/EN300-05436,30,"12bisElectrophiles, AlkylHalides","aHaloKetones, Alkyl_halides",O=C(CCl)c1ccc[nH]1,False,11_24_EnamineRush_42,9,143.573,1,1.43620,2,32.86,True,1,0,False,1
1,171.62,"2-chloro-1-(2,5-dimethyl-1H-pyrrol-3-yl)ethan-...",https://www.enaminestore.com/catalog/EN300-27026,30,"12bisElectrophiles, AlkylHalides","aHaloKetones, Alkyl_halides",Cc1cc(C(=O)CCl)c(C)[nH]1,False,11_24_EnamineRush_66,11,171.627,1,2.05304,2,32.86,True,1,0,False,1
2,163.01,2-bromo-1-cyclopropylethan-1-one,https://www.enaminestore.com/catalog/EN300-57214,300,"12bisElectrophiles, AlkylHalides","aHaloKetones, Alkyl_halides",O=C(CBr)C1CC1,False,11_24_EnamineRush_168,7,163.014,0,1.36040,2,17.07,True,0,0,False,1
3,120.58,1-chloropentan-2-one,https://www.enaminestore.com/catalog/EN300-59809,30,"12bisElectrophiles, AlkylHalides","aHaloKetones, Alkyl_halides",CCCC(=O)CCl,False,11_24_EnamineRush_178,7,120.579,0,1.59440,3,17.07,True,0,0,False,1
4,165.03,1-bromo-3-methylbutan-2-one,https://www.enaminestore.com/catalog/EN300-72257,30,"12bisElectrophiles, AlkylHalides","aHaloKetones, Alkyl_halides",CC(C)C(=O)CBr,False,11_24_EnamineRush_182,7,165.030,0,1.60640,2,17.07,True,0,0,False,1


In [15]:
smiles_list = bb_df["SMILES"].tolist()
ids = bb_df["ID"].tolist()

clean = []
for i in tqdm(range(0, len(smiles_list), 10_000), desc="Cleaning"):
    chunk = smiles_list[i : i + 10_000]
    with multiprocessing.Pool(NUM_CPUS) as pool:
        clean.extend(pool.map(get_clean_smiles, chunk))

out = Path(f"../building_blocks/aqemia_{NAME}.smi.gz")
out.parent.mkdir(parents=True, exist_ok=True)
n_written = 0
with gzip.open(out, "wt") as w:
    for smi, id_ in zip(clean, ids, strict=True):
        if smi is not None:
            w.write(f"{smi}\t{id_}\n")
            n_written += 1
print(f"Wrote {n_written}/{len(smiles_list)} blocks to {out}")

Cleaning:  33%|███▎      | 1/3 [00:00<00:01,  1.60it/s][12:15:50] Explicit valence for atom # 2 O, 2, is greater than permitted
[12:15:50] Explicit valence for atom # 2 O, 2, is greater than permitted
Cleaning: 100%|██████████| 3/3 [00:01<00:00,  1.83it/s]

Wrote 23605/23789 blocks to ../building_blocks/aqemia_2024_11_simple_bbs_24K.smi.gz


### Reactions : parquet to `.txt`

In [16]:
RXN_NAME = "2025_02_03_default_reaction_templates"
rxn_path = f"s3://aqemia-datahub/forward_synthesis/aqemia_reactions_datasets/{RXN_NAME}.parquet"
reaction_df = s3.read_parquet(rxn_path)
print(f"Loaded {len(reaction_df)} templates. Columns: {list(reaction_df.columns)}")

out_tpl = Path(f"../templates/aqemia_{RXN_NAME}.txt")
out_tpl.parent.mkdir(parents=True, exist_ok=True)
with out_tpl.open("w") as w:
    for smarts in reaction_df["template_SMARTS"]:
        w.write(f"{smarts.strip()}\n")
print(f"Wrote {len(reaction_df)} templates to {out_tpl}")

Loaded 180 templates. Columns: ['template_SMARTS', 'id', 'id_with_permutation', 'index', 'bb1_type', 'bb2_type', 'phase', 'is_symmetry']
Wrote 180 templates to ../templates/aqemia_2025_02_03_default_reaction_templates.txt
